# Print Network to PDF Example

This notebook demonstrates the `print_network` functionality which generates IEEE-standard electrical diagrams from Power Grid Model network data and exports them to PDF.

## Key Features

The `print_network` function creates professional electrical diagrams with:

### IEEE-Standard Symbols
- **Buses/Nodes**: Thick horizontal bars
- **Lines**: Connecting lines between nodes  
- **Transformers**: Two circles (for two-winding)
- **Sources**: Circle with sine wave (~)
- **Loads**: Triangle pointing down
- **Generators**: Circle with 'G'
- **Shunts**: Capacitor symbol

### Orthogonal Routing (NEW!)
- **Grid-aligned nodes**: All components snap to a grid for professional appearance
- **Manhattan routing**: Connections use only horizontal and vertical segments (no diagonals!)
- **Right-angle turns**: Follows electrical engineering drawing standards
- **Professional output**: Matches IEEE standard electrical diagrams

## 1. Print diagram from existing PGM JSON data

Let's load the tiny-net example and generate a PDF diagram.

In [ ]:
from power_grid_model_io.converters import PgmJsonConverter
from power_grid_model_io.utils.print_network import print_network

# Load network data
source_file = "data/tiny-net/input.json"
converter = PgmJsonConverter(source_file=source_file)
input_data, extra_info = converter.load_input_data()

# Print network components
print("Network components:")
for component_type, data in input_data.items():
    print(f"  {component_type}: {len(data)} items")

In [ ]:
# Generate PDF diagram with orthogonal routing
output_file = "tiny_net_diagram.pdf"
print_network(input_data, output_file)
print(f"✓ Network diagram saved to {output_file}")
print("\nFeatures in the generated PDF:")
print("  • Nodes aligned to grid")
print("  • Orthogonal (right-angle) connections")
print("  • IEEE-standard component symbols")

## 2. Create a custom network and print it

Let's create a simple network from scratch to demonstrate the function.

In [ ]:
import numpy as np

# Create a simple radial network:
# Source -> Node1 -> Line1 -> Node2 -> Line2 -> Node3
#                                |                |
#                             Load1            Load2

custom_network = {
    "node": np.array(
        [
            (1, 110000.0),  # High voltage node
            (2, 10500.0),   # Medium voltage node
            (3, 10500.0),   # Medium voltage node
        ],
        dtype=[("id", "i4"), ("u_rated", "f8")],
    ),
    "transformer": np.array(
        [
            # All 29 fields: id, from_node, to_node, from_status, to_status, u1, u2, sn, uk, pk, i0, p0,
            # winding_from, winding_to, clock, tap_side, tap_pos, tap_min, tap_max, tap_nom, tap_size,
            # uk_min, uk_max, pk_min, pk_max, r_grounding_from, x_grounding_from, r_grounding_to, x_grounding_to
            (100, 1, 2, 1, 1, 110000.0, 10500.0, 40e6, 0.0, 0.1, 0.0, 5e-6, 0, 1, 1, 0, 0, 0, 0, 0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0),
        ],
        dtype=[
            ("id", "i4"), ("from_node", "i4"), ("to_node", "i4"),
            ("from_status", "i1"), ("to_status", "i1"),
            ("u1", "f8"), ("u2", "f8"), ("sn", "f8"),
            ("uk", "f8"), ("pk", "f8"), ("i0", "f8"), ("p0", "f8"),
            ("winding_from", "i4"), ("winding_to", "i4"),
            ("clock", "i4"), ("tap_side", "i4"),
            ("tap_pos", "i4"), ("tap_min", "i4"), ("tap_max", "i4"),
            ("tap_nom", "i4"), ("tap_size", "f8"),
            ("uk_min", "f8"), ("uk_max", "f8"),
            ("pk_min", "f8"), ("pk_max", "f8"),
            ("r_grounding_from", "f8"), ("x_grounding_from", "f8"),
            ("r_grounding_to", "f8"), ("x_grounding_to", "f8"),
        ],
    ),
    "line": np.array(
        [
            (200, 2, 3, 1, 1, 0.25, 0.30, 5e-05, 0.1, 600.0),
        ],
        dtype=[
            ("id", "i4"), ("from_node", "i4"), ("to_node", "i4"),
            ("from_status", "i1"), ("to_status", "i1"),
            ("r1", "f8"), ("x1", "f8"), ("c1", "f8"),
            ("tan1", "f8"), ("i_n", "f8"),
        ],
    ),
    "source": np.array(
        [(1000, 1, 1, 1.02, 1e10, 0.0)],
        dtype=[
            ("id", "i4"), ("node", "i4"), ("status", "i1"),
            ("u_ref", "f8"), ("sk", "f8"), ("rx_ratio", "f8"),
        ],
    ),
    "sym_load": np.array(
        [
            (2000, 2, 1, 4),  # Load at node 2
            (2001, 3, 1, 4),  # Load at node 3
        ],
        dtype=[("id", "i4"), ("node", "i4"), ("status", "i1"), ("type", "i4")],
    ),
}

print("Custom network created with:")
for component_type, data in custom_network.items():
    print(f"  {component_type}: {len(data)} items")

In [ ]:
# Generate PDF for custom network
output_file = "custom_network_diagram.pdf"
print_network(custom_network, output_file)
print(f"Custom network diagram saved to {output_file}")

## 3. Using with PandaPower networks

The function also works with networks converted from PandaPower.

In [ ]:
# Note: This requires pandapower to be installed
try:
    from power_grid_model_io.converters import PandaPowerConverter
    import pandapower as pp
    import pandapower.networks as pn

    # Load a standard PandaPower network
    pp_net = pn.case9()
    
    print(f"PandaPower network: {pp_net.name}")
    print(f"  Buses: {len(pp_net.bus)}")
    print(f"  Lines: {len(pp_net.line)}")
    print(f"  Transformers: {len(pp_net.trafo)}")
    print(f"  Loads: {len(pp_net.load)}")
    print(f"  Generators: {len(pp_net.gen)}")
    
    # Convert to PGM format
    converter = PandaPowerConverter()
    pgm_data = converter.load_input_data(pp_net)
    
    # Generate PDF
    output_file = "pandapower_case9_diagram.pdf"
    print_network(pgm_data, output_file)
    print(f"\nPandaPower network diagram saved to {output_file}")
    
except ImportError:
    print("PandaPower not installed. Skipping this example.")

## Summary

The `print_network` function provides an easy way to visualize Power Grid Model networks as professional electrical diagrams:

- **Automatic layout**: Uses graph algorithms to position components intelligently
- **IEEE standards**: Components are drawn with standard electrical symbols
- **PDF export**: High-quality vector output suitable for documentation
- **Flexible input**: Works with PGM JSON, PandaPower, or manually created data

The generated PDFs can be used for:
- Documentation and reports
- Network analysis and planning
- Educational materials
- Presentations and proposals